# Model Interpretation: Real / Fake, Gender and Age Group (with Grad-CAM)

Give **one picture** (or let the notebook pick one real and one fake test image) and it shows:

1. **Real or fake?** (with confidence and a Grad-CAM heatmap)
2. **Which gender?** (only for images judged real; with Grad-CAM)
3. **Which age group?** (only for images judged real; with Grad-CAM)

For an image judged **fake**, gender is shown as *Unknown* and age as *Not applicable*. This is the same rule used for the fake images in the dataset.

Three separately trained models are used, one per task (see the settings cell). Grad-CAM shows **where a model looked** for its answer (red = strong influence). It does not prove the answer is correct.

### Import Libraries:

In [1]:
import io
import random
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pathlib import Path
from PIL import Image
from torchvision import transforms, models

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


### Settings:

Change only the values in this cell.

In [2]:
# --------------------------------------------------
# Change only these values
# --------------------------------------------------

PROJECT_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final")
MODELS_DIR = PROJECT_DIR / "models"
DATA_DIR = PROJECT_DIR / "data"
CHART_DIR = PROJECT_DIR/"charts"/"eda"

# The two pictures that are analysed: one REAL and one FAKE.
# None -> a random image from the real_fake test set is used.
# Or give the full path of your own picture.
REAL_IMAGE_PATH = None       # example: r"C:\Users\jagad\Pictures\my_photo.jpg"
FAKE_IMAGE_PATH = None

# If the confidence is below this value, the answer is shown as "Unknown".
GENDER_MIN_CONF = 0.60
AGE_MIN_CONF = 0.40          # lower, because age has more classes and is harder

SEED = None                  # None -> different random images every run; a number -> the same ones


TASKS = {
    "real_fake": {
        "arch": "resnet50",
        "file": MODELS_DIR / "real_fake" / "resnet50_model_real_fake.pth",
        "classes": ["fake", "real"],
        "prep": "norm",
        "exclude": [],
    },
    "gender": {
        "arch": "vit_b16",
        "file": MODELS_DIR / "gender" / "vit_model_gender.pth",
        "classes": ["female", "male", "unknown"],   # the saved gender model has 3 outputs
        "prep": "norm",
        "exclude": ["unknown"],                     # "unknown" is never predicted for a real photo
    },
    "age": {
        "arch": "resnet50",
        "file": MODELS_DIR / "age_group" / "resnet50_age_group.pth",
        "classes": ["18-25", "26-35", "36-50", "50+", "not_applicable", "under_18"],
        "prep": "raw",
        "exclude": ["not_applicable"],
    },
}

# Test folders used to pick random example images
TEST_DIRS = {
    "real_fake": DATA_DIR / "split_real_fake" / "test",
    "gender": DATA_DIR / "split_gender" / "test",
    "age": DATA_DIR / "split_age" / "test",
}

if SEED is not None:
    random.seed(SEED)

### Image preparation:

Each model gets the image prepared the same way as its training images. After that, every model uses the same resize to 224 x 224 and the same normalization as the validation and test images.

In [3]:
def prepare_norm(img):
    """Centre-crop to a square, resize to 256 x 256, save as JPEG quality 95 (as for the training images)."""
    img = img.convert("RGB")
    w, h = img.size
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    img = img.crop((left, top, left + s, top + s)).resize((256, 256), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=95)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def prepare_raw(img):
    """The image as it is."""
    return img.convert("RGB")


PREP = {"norm": prepare_norm, "raw": prepare_raw}

to_tensor = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

### Load the three models and prepare Grad-CAM:

The model structures are rebuilt exactly as in the training notebooks, then the saved weights are loaded.
Grad-CAM uses the last convolutional block of each CNN. The Vision Transformer (ViT) has no convolutions, so it uses the layer norm of its last encoder block, and its tokens are reshaped back into a 14 x 14 grid.

In [4]:

class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))

        self.block1 = block(3, 32)
        self.block2 = block(32, 64)
        self.block3 = block(64, 128)
        self.block4 = block(128, 256)
        self.block5 = block(256, 512)
        self.dropout_conv = nn.Dropout(0.3)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(512, 256)
        self.dropout_fc = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.dropout_conv(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.dropout_fc(x)
        return self.fc2(x)


def build_model(arch, n):
    if arch == "cnn":
        return CNN(n)
    if arch == "resnet50":
        m = models.resnet50(weights=None)
        m.fc = nn.Sequential(nn.Linear(m.fc.in_features, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, n))
        return m
    if arch == "vgg16":
        m = models.vgg16(weights=None)
        m.classifier = nn.Sequential(nn.Linear(25088, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, n))
        return m
    if arch == "efficientnet_b0":
        m = models.efficientnet_b0(weights=None)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, 256),
                                     nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, n))
        return m
    if arch == "vit_b16":
        m = models.vit_b_16(weights=None)
        m.heads = nn.Sequential(nn.Linear(m.heads[0].in_features, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, n))
        return m
    raise ValueError(f"Unknown architecture: {arch}")


def get_target_layers(model, arch):
    """The layer that Grad-CAM explains."""
    if arch == "cnn":
        return [model.block5[0]]
    if arch == "resnet50":
        return [model.layer4[-1]]
    if arch == "vgg16":
        return [model.features[-1]]
    if arch == "efficientnet_b0":
        return [model.features[-1]]
    if arch == "vit_b16":
        return [model.encoder.layers[-1].ln_1]
    raise ValueError(f"Unknown architecture: {arch}")


def vit_reshape_transform(tensor, height=14, width=14):
    """ViT tokens (1 class token + 14 x 14 patches) -> a feature map (channels, 14, 14)."""
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    return result.transpose(2, 3).transpose(1, 2)


HEAD_PREFIX = {"cnn": "fc2", "resnet50": "fc", "vgg16": "classifier",
               "efficientnet_b0": "classifier", "vit_b16": "heads"}


def read_state_dict(path):
    """Load a .pth file and return a clean state_dict (handles wrapped checkpoints and 'module.' prefixes)."""
    obj = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(obj, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in obj and isinstance(obj[key], dict):
                obj = obj[key]
                break
    if not isinstance(obj, dict):
        raise TypeError(f"{path} does not contain a state_dict (found {type(obj).__name__}). "
                        "It was probably saved with torch.save(model); re-save with model.state_dict().")
    return {k.replace("module.", "", 1): v for k, v in obj.items()}


def count_outputs(arch, state_dict):
    """Number of outputs of the last layer of the model head, read from the saved weights."""
    prefix = HEAD_PREFIX[arch]
    weights = [k for k in state_dict if k.startswith(prefix) and k.endswith(".weight")]
    if not weights:
        raise KeyError(f"No '{prefix}...weight' keys found for arch '{arch}'.")
    # the last layer of the head = the key with the highest layer number
    last = max(weights, key=lambda k: [int(t) if t.isdigit() else -1 for t in k.split(".")])
    return state_dict[last].shape[0], last


LOADED = {}

for task, cfg in TASKS.items():
    if not cfg["file"].exists():
        raise FileNotFoundError(f"Model file for '{task}' not found: {cfg['file']}")

    state_dict = read_state_dict(cfg["file"])

    try:
        n_classes, last_key = count_outputs(cfg["arch"], state_dict)
    except KeyError as e:
        keys = list(state_dict.keys())
        raise KeyError(
            f"Task '{task}' (arch='{cfg['arch']}', file={cfg['file'].name}): {e}\n"
            f"The file does not look like a {cfg['arch']} model. Check 'arch' and 'file' for this task in the "
            f"settings cell.\nLast keys in the file: {keys[-4:]}"
        ) from None

    # Safety check: the class list in TASKS must match the saved model
    if n_classes != len(cfg["classes"]):
        raise ValueError(
            f"'{task}': the model file has {n_classes} outputs but TASKS lists "
            f"{len(cfg['classes'])} classes {cfg['classes']}. Fix 'classes' in the settings cell "
            "(use the class_names printed in that task's training notebook)."
        )

    # Build the model. The head is the 256-unit head used in training, unless the file holds a plain linear head.
    model = build_model(cfg["arch"], n_classes)
    if last_key == HEAD_PREFIX[cfg["arch"]] + ".weight" and cfg["arch"] == "resnet50":
        model.fc = nn.Linear(model.fc[0].in_features, n_classes)      # plain head: fc.weight / fc.bias
    try:
        model.load_state_dict(state_dict)
    except RuntimeError as e:
        raise RuntimeError(f"Task '{task}': the saved weights do not match arch '{cfg['arch']}'.\n{e}") from None
    model = model.to(device).eval()

    cam = GradCAM(
        model=model,
        target_layers=get_target_layers(model, cfg["arch"]),
        reshape_transform=vit_reshape_transform if cfg["arch"] == "vit_b16" else None,
    )

    LOADED[task] = {**cfg, "model": model, "cam": cam, "num_classes": n_classes}
    print(f"{task:10s} {cfg['arch']:16s} classes = {n_classes} -> loaded")


real_fake  resnet50         classes = 2 -> loaded
gender     vit_b16          classes = 3 -> loaded
age        resnet50         classes = 6 -> loaded


### Predict and explain one image with one model:

For each task the model gives a probability for every class. Grad-CAM then explains the **predicted** class.
For the age model, the class `not_applicable` is never chosen for a real photo (it is mostly the label of fake images in the dataset), and the other probabilities are rescaled to add up to 1.

In [5]:
def analyse(task, pil_image):
    cfg = LOADED[task]

    prepared = PREP[cfg["prep"]](pil_image)
    x = to_tensor(prepared).unsqueeze(0).to(device)


    # Probabilities
    with torch.no_grad():
        probs = torch.softmax(cfg["model"](x), dim=1)[0].cpu()

    for name in cfg["exclude"]:
        probs[cfg["classes"].index(name)] = 0.0
    probs = probs / probs.sum()

    idx = int(probs.argmax())

    # Grad-CAM for the predicted class
    gray = cfg["cam"](input_tensor=x, targets=[ClassifierOutputTarget(idx)])[0]

    # The image exactly as the model saw it (undo the normalization)
    rgb = x[0].cpu().permute(1, 2, 0).numpy()
    rgb = np.clip(rgb * IMAGENET_STD + IMAGENET_MEAN, 0, 1).astype(np.float32)

    return {
        "task": task,
        "label": cfg["classes"][idx],
        "confidence": float(probs[idx]),
        "probs": {c: float(p) for c, p in zip(cfg["classes"], probs)},
        "original": (rgb * 255).astype(np.uint8),
        "gradcam": show_cam_on_image(rgb, gray, use_rgb=True),
    }

### Full prediction for one picture:

1. The real/fake model decides.
2. **Fake** -> gender *Unknown*, age *Not applicable*.
3. **Real** -> the gender model and the age model run, each with its own Grad-CAM.
4. If the confidence of gender or age is below the limit in the settings, that answer is shown as *Unknown*.

In [6]:
try:
    TRUTH = pd.read_csv(DATA_DIR / "final_image_dataset.csv", dtype=str).set_index("image_id")
except FileNotFoundError:
    TRUTH = None


def true_labels(path):
    """If the picture is one of the dataset images, return its known labels."""
    if TRUTH is not None and Path(path).stem in TRUTH.index:
        r = TRUTH.loc[Path(path).stem]
        return f"{r['label']} | gender: {r['gender']} | age group: {r['age_group']}"
    return None


def predict_all(pil_image):
    """Run the three models on one picture. Returns (results, gender_text, age_text)."""
    results = {"real_fake": analyse("real_fake", pil_image)}
    rf = results["real_fake"]

    if rf["label"] == "fake":
        gender_text, age_text = "Unknown", "Not applicable"
    else:
        results["gender"] = analyse("gender", pil_image)
        results["age"] = analyse("age", pil_image)
        g, a = results["gender"], results["age"]
        gender_text = g["label"].capitalize() if g["confidence"] >= GENDER_MIN_CONF else "Unknown"
        age_text = a["label"] if a["confidence"] >= AGE_MIN_CONF else "Unknown"

    return results, gender_text, age_text


def print_summary(results, gender_text, age_text):
    rf = results["real_fake"]
    rows = [{"Question": "Real or fake?", "Answer": rf["label"].upper(), "Confidence": f"{rf['confidence']:.1%}"}]
    if "gender" in results:
        rows.append({"Question": "Gender", "Answer": gender_text, "Confidence": f"{results['gender']['confidence']:.1%}"})
        rows.append({"Question": "Age group", "Answer": age_text, "Confidence": f"{results['age']['confidence']:.1%}"})
    else:
        rows.append({"Question": "Gender", "Answer": gender_text, "Confidence": "rule (fake image)"})
        rows.append({"Question": "Age group", "Answer": age_text, "Confidence": "rule (fake image)"})
    print(pd.DataFrame(rows).to_string(index=False))


def full_prediction(pil_image):
    """One picture -> printed summary + a figure with a Grad-CAM for each model that ran."""
    results, gender_text, age_text = predict_all(pil_image)
    rf = results["real_fake"]
    print_summary(results, gender_text, age_text)

    panels = [
        (rf["original"], "Image (as seen by the real/fake model)"),
        (rf["gradcam"], f"Real/fake Grad-CAM: {rf['label'].upper()} ({rf['confidence']:.0%})"),
    ]
    if "gender" in results:
        g, a = results["gender"], results["age"]
        panels.append((g["gradcam"], f"Gender Grad-CAM: {g['label']} ({g['confidence']:.0%})"))
        panels.append((a["gradcam"], f"Age Grad-CAM: {a['label']} ({a['confidence']:.0%})"))

    fig = make_subplots(rows=1, cols=len(panels), subplot_titles=[p[1] for p in panels],
                        horizontal_spacing=0.015)
    for k, (img, _) in enumerate(panels, start=1):
        fig.add_trace(go.Image(z=img), row=1, col=k)

    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    fig.update_annotations(font_size=11)
    fig.update_layout(
        title=f"{rf['label'].upper()}  |  Gender: {gender_text}  |  Age group: {age_text}",
        width=300 * len(panels) + 40,
        height=500
    )
    return fig, results


### Run it: one REAL and one FAKE image

One real and one fake test image are analysed and shown in **one figure**: the top row is the real example, the bottom row is the fake example. For each image you see the picture, the real/fake Grad-CAM, and (for an image judged real) the gender and age Grad-CAM. For the fake one, gender is *Unknown* and age is *Not applicable*. If the dataset's known labels exist for a picture, they are printed too.

Set `REAL_IMAGE_PATH` / `FAKE_IMAGE_PATH` in the settings cell to use your own pictures; with `None` a random test image is used. The figure is always saved as `grad_cam_fig.pkl` in the charts folder (`CHART_DIR`).

In [7]:
# --------------------------------------------------
# 1. Pick ONE real and ONE fake test image
# --------------------------------------------------
image_extensions = {".jpg", ".jpeg", ".png"}


def pick_image(kind, given_path):
    """The path you gave, or a random image from the real_fake test folder of this kind."""
    if given_path is not None:
        return Path(given_path)
    folder = TEST_DIRS["real_fake"] / kind
    files = [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in image_extensions]
    if not files:
        raise FileNotFoundError(f"No images found in directory: {folder}. Please check your path.")
    return random.choice(files)


examples = [
    ("real", pick_image("real", REAL_IMAGE_PATH)),
    ("fake", pick_image("fake", FAKE_IMAGE_PATH)),
]

# --------------------------------------------------
# 2. Predict + Grad-CAM for both
# --------------------------------------------------
all_results = []      # (image path, results) - used by the probability cell below
rows_data = []        # what is drawn in the figure

for kind, image_path in examples:
    print("=" * 70)
    print(f"{kind.upper()} example: {image_path}")

    known = true_labels(image_path)
    if known:
        print("Known labels in the dataset ->", known)
    print()

    picture = Image.open(image_path).convert("RGB")
    if min(picture.size) < 256:
        print("Warning: this image is smaller than 256 px. The models were trained on larger images.\n")

    results, gender_text, age_text = predict_all(picture)
    print_summary(results, gender_text, age_text)
    print()

    all_results.append((image_path, results))
    rows_data.append((kind, results, gender_text, age_text))

# --------------------------------------------------
# 3. ONE figure: top row = REAL example, bottom row = FAKE example
#    columns: image | real/fake Grad-CAM | gender Grad-CAM | age Grad-CAM
# --------------------------------------------------
subplot_titles = []
for kind, results, gender_text, age_text in rows_data:
    rf = results["real_fake"]
    subplot_titles.append(f"{kind.upper()} test image")
    subplot_titles.append(f"Real/fake Grad-CAM: {rf['label'].upper()}")
    if "gender" in results:
        g, a = results["gender"], results["age"]
        subplot_titles.append(f"Gender Grad-CAM: {g['label']}")
        subplot_titles.append(f"Age Grad-CAM: {a['label']}")
    else:
        subplot_titles.append("Gender: Unknown (fake image)")
        subplot_titles.append("Age group: Not applicable (fake image)")

fig = make_subplots(rows=2, cols=4, subplot_titles=subplot_titles,
                    vertical_spacing=0.12, horizontal_spacing=0.015)

for row_number, (kind, results, gender_text, age_text) in enumerate(rows_data, start=1):
    rf = results["real_fake"]
    fig.add_trace(go.Image(z=rf["original"]), row=row_number, col=1)
    fig.add_trace(go.Image(z=rf["gradcam"]), row=row_number, col=2)
    if "gender" in results:
        fig.add_trace(go.Image(z=results["gender"]["gradcam"]), row=row_number, col=3)
        fig.add_trace(go.Image(z=results["age"]["gradcam"]), row=row_number, col=4)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_annotations(font_size=11)
fig.update_layout(
    title="Grad-CAM: one REAL image (top row) and one FAKE image (bottom row)",
    width=300 * 4 + 40,
    height=760
)

fig.show()

# --------------------------------------------------
# 4. Always save the figure in the charts folder (CHART_DIR)
# --------------------------------------------------
CHART_DIR = Path(CHART_DIR)
CHART_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(fig, CHART_DIR / "grad_cam_fig.pkl")
print("grad_cam_fig saved as pkl file in:", CHART_DIR)


REAL example: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\split_real_fake\test\real\2496.jpg
Known labels in the dataset -> real | gender: female | age group: 50+

     Question Answer Confidence
Real or fake?   REAL     100.0%
       Gender Female      77.7%
    Age group    50+      45.6%

FAKE example: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\split_real_fake\test\fake\5413.jpg
Known labels in the dataset -> fake | gender: unknown | age group: not_applicable

     Question         Answer        Confidence
Real or fake?           FAKE            100.0%
       Gender        Unknown rule (fake image)
    Age group Not applicable rule (fake image)



grad_cam_fig saved as pkl file in: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\charts\eda
